In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import scipy.stats as stats
from matplotlib import pyplot as plt

In [ ]:
df = pd.read_csv('../data/short_ships(in).csv')
df = df[df["SHIPS_TIME"] <= 60]
df = df[df["SHIPS_TIME"] >= 0]

,SHIPS_case,SHIPS_time,SHIPS_VMAX,SHIPS_stormyear,SHIPS_stormbasin,SHIPS_stormid,SHIPS_TIME,SHIPS_MSLP,SHIPS_VMPI,SHIPS_LAT,SHIPS_LON
2,0,0 days 00:00:00,20.0,1982,AL,AL011982,0,1005.0,123.0,21.7,-87.1
3,0,0 days 06:00:00,25.0,1982,AL,AL011982,6,1004.0,122.0,22.0,-86.3
4,0,0 days 12:00:00,30.0,1982,AL,AL011982,12,1003.0,117.0,22.3,-85.6
5,0,0 days 18:00:00,40.0,1982,AL,AL011982,18,1001.0,117.0,22.3,-84.9
6,0,1 days 00:00:00,50.0,1982,AL,AL011982,24,995.0,121.0,22.7,-83.9
7,0,1 days 06:00:00,75.0,1982,AL,AL011982,30,985.0,105.0,23.6,-83.0
8,0,1 days 12:00:00,65.0,1982,AL,AL011982,36,992.0,100.0,24.6,-82.5
9,0,1 days 18:00:00,55.0,1982,AL,AL011982,42,998.0,107.0,25.8,-83.9
10,0,2 days 00:00:00,45.0,1982,AL,AL011982,48,1002.0,114.0,26.0,-84.8
11,0,2 days 06:00:00,40.0,1982,AL,AL011982,54,1005.0,102.0,23.8,-84.0


In [4]:
# find vmax over the first 12 hours for each storm case
VMAX12_df = df[df["SHIPS_TIME"] <= 12][["SHIPS_case", "SHIPS_VMAX"]].groupby("SHIPS_case").max(min_count=3).rename(columns={"SHIPS_VMAX": "VMAX12"})
VMAX12_df

,VMAX12
SHIPS_case,
0,30.0
1,40.0
2,50.0
3,75.0
4,75.0
...,...
31998,40.0
31999,35.0
32000,35.0


In [5]:
MSLP12_df = df[df["SHIPS_TIME"] <= 12][["SHIPS_case", "SHIPS_MSLP"]].groupby("SHIPS_case").min(min_count=3).rename(columns={"SHIPS_MSLP": "MSLP12"})
MSLP12_df

,MSLP12
SHIPS_case,
0,1003.0
1,1001.0
2,995.0
3,985.0
4,985.0
...,...
31998,1005.0
31999,1006.0
32000,1006.0


In [6]:
POT_df = df[df["SHIPS_TIME"] == 12][["SHIPS_case", "SHIPS_VMPI", "SHIPS_VMAX"]]
POT_df["POT12"] = POT_df["SHIPS_VMPI"] - POT_df["SHIPS_VMAX"]
POT_df = POT_df[["SHIPS_case", "POT12"]].set_index("SHIPS_case")
POT_df

,POT12
SHIPS_case,
0,87.0
1,77.0
2,71.0
3,30.0
4,35.0
...,...
31998,104.0
31999,111.0
32000,105.0


In [9]:
VMAX60_df = df[df["SHIPS_TIME"] > 12][["SHIPS_case", "SHIPS_VMAX"]].groupby("SHIPS_case").max(min_count=8).rename(columns={"SHIPS_VMAX": "VMAX60"})
VMAX60_df

,VMAX60
SHIPS_case,
0,75.0
1,75.0
2,75.0
3,65.0
4,55.0
...,...
31998,NaN
31999,NaN
32000,NaN


In [23]:
cases = df[["SHIPS_case", "SHIPS_stormyear", "SHIPS_stormbasin", "SHIPS_stormid"]].groupby("SHIPS_case").first().rename(columns={"SHIPS_stormyear": "Year", "SHIPS_stormbasin": "Basin", "SHIPS_stormid": "StormID"})
cases

,Year,Basin,StormID
SHIPS_case,,,
0,1982,AL,AL011982
1,1982,AL,AL011982
2,1982,AL,AL011982
3,1982,AL,AL011982
4,1982,AL,AL011982
...,...,...,...
31998,2019,CP,CP012019
31999,2019,CP,CP012019
32000,2019,CP,CP012019


In [33]:
data = cases.join(VMAX12_df, on="SHIPS_case").join(MSLP12_df, on="SHIPS_case").join(POT_df, on="SHIPS_case").join(VMAX60_df, on="SHIPS_case")
data = data.groupby("StormID").first().dropna()
data["Basin"].value_counts()

Basin
EP    481
AL    456
CP     16
Name: count, dtype: int64